In [1]:
import pandas as pd

df = pd.read_csv("data/Keylogger_Detection.csv")
print(df.shape)
df.head()

(523617, 86)


/var/folders/j_/x0rzqtss1wgdv8tmm18x55mw0000gn/T/ipykernel_4669/3643055495.py:3: DtypeWarning: Columns (48,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/Keylogger_Detection.csv")


,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Class
0,0,10.42.0.211-52.6.25.230-34451-443-6,10.42.0.211,34451.0,52.6.25.230,443.0,6.0,04/08/2017 05:12:36,12140931.0,9.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,1,172.217.3.99-10.42.0.151-443-53892-6,10.42.0.151,53892.0,172.217.3.99,443.0,6.0,04/08/2017 07:55:51,418882.0,102.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,2,172.217.3.98-10.42.0.151-443-50750-6,172.217.3.98,443.0,10.42.0.151,50750.0,6.0,04/08/2017 08:48:19,45.0,2.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,3,10.42.0.211-10.42.0.1-23025-53-17,10.42.0.211,23025.0,10.42.0.1,53.0,17.0,04/08/2017 05:54:10,541699.0,1.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,4,10.42.0.211-123.129.244.226-52602-443-6,10.42.0.211,52602.0,123.129.244.226,443.0,6.0,04/08/2017 08:44:25,7310795.0,3.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [2]:
df = pd.read_csv("data/Keylogger_Detection.csv", low_memory=False)
print(df.columns[-5:].tolist())
print(df.columns[[48, 56]].tolist())
print(df.iloc[:, -1].value_counts())

['Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min', 'Class']
[' Packet Length Std', ' CWE Flag Count']
Class
Benign       308813
Keylogger    214804
Name: count, dtype: int64


In [3]:
df.columns = df.columns.str.strip()

print("Valores em falta:", df.isna().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())
print(df.dtypes.value_counts())

for c in ["Packet Length Std", "CWE Flag Count"]:
    invalidos = pd.to_numeric(df[c], errors="coerce").isna() & df[c].notna()
    print(c, "-> valores não numéricos:", invalidos.sum(), df.loc[invalidos, c].unique()[:5])

Valores em falta: 918
Linhas duplicadas: 0
float64    78
object      7
int64       1
Name: count, dtype: int64
Packet Length Std -> valores não numéricos: 2 ['SCAREWARE']
CWE Flag Count -> valores não numéricos: 3 ['SCAREWARE']


In [4]:
mask = (df["Packet Length Std"] == "SCAREWARE") | (df["CWE Flag Count"] == "SCAREWARE")
print(df.loc[mask, ["Flow ID", "Source IP", "Packet Length Std", "CWE Flag Count", "Class"]])

faltas = df.isna().sum()
print(faltas[faltas > 0])


       Flow ID Source IP Packet Length Std CWE Flag Count      Class
59378      NaN         0         SCAREWARE            NaN  Keylogger
121271     NaN     281.0               0.0      SCAREWARE  Keylogger
463309     NaN         0         SCAREWARE            NaN  Keylogger
488826     NaN     281.0               0.0      SCAREWARE  Keylogger
499706     NaN     281.0               0.0      SCAREWARE  Keylogger
Flow ID           7
Flow IAT Mean     3
Flow IAT Std      3
Flow IAT Max      3
Flow IAT Min      3
                 ..
Active Min       22
Idle Mean        22
Idle Std         22
Idle Max         22
Idle Min         22
Length: 63, dtype: int64


In [5]:
import numpy as np

linhas_com_nan = df.isna().any(axis=1)
print("Linhas com pelo menos 1 valor em falta:", linhas_com_nan.sum())
print(df.loc[linhas_com_nan, "Class"].value_counts())

num = df.select_dtypes(include="number")
print("Valores infinitos:", np.isinf(num).sum().sum())

Linhas com pelo menos 1 valor em falta: 24
Class
Keylogger    24
Name: count, dtype: int64
Valores infinitos: 0


In [6]:
antes = len(df)

# 1. "SCAREWARE" passa a NaN
for c in ["Packet Length Std", "CWE Flag Count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 2. Remover linhas com valores em falta
df = df.dropna()

# 3. Remover colunas de identificação
df = df.drop(columns=["Unnamed: 0", "Flow ID", "Source IP", "Destination IP", "Timestamp"])

print("Linhas removidas:", antes - len(df))
print("Shape:", df.shape)
print("Duplicadas:", df.duplicated().sum())
print(df.dtypes.value_counts())
print(df["Class"].value_counts())

Linhas removidas: 24
Shape: (523593, 81)
Duplicadas: 357714
float64    80
object      1
Name: count, dtype: int64
Class
Benign       308813
Keylogger    214780
Name: count, dtype: int64


In [7]:
feats = [c for c in df.columns if c != "Class"]

sem_dup = df.drop_duplicates()
sem_dup_feats = df.drop_duplicates(subset=feats)

print("Únicas (features + Class):", len(sem_dup))
print("Únicas (só features):", len(sem_dup_feats))
print(sem_dup["Class"].value_counts())

Únicas (features + Class): 165879
Únicas (só features): 165879
Class
Benign       97856
Keylogger    68023
Name: count, dtype: int64


In [8]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape final:", df.shape)

df.to_csv("data/keylogger_clean.csv", index=False)


Shape final: (165879, 81)
